# Day 20 Capstone Mini-Project
## Image Classifier with Transfer Learning

CIFAR-10 → Baseline CNN → Data Augmentation → MobileNetV2 → Fine-Tuning → Evaluation → Grad-CAM → Reflection

## Step 0 — Imports and Offline CIFAR-10 Setup

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

RNG = 42
np.random.seed(RNG)
tf.random.set_seed(RNG)

(x_train_full, y_train_full), (x_test_full, y_test_full) = keras.datasets.cifar10.load_data()

CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

def subset(x, y, per_class):
    y = y.ravel()
    idx = np.hstack([
        np.random.RandomState(RNG).choice(
            np.where(y == c)[0],
            per_class,
            replace=False
        )
        for c in range(10)
    ])

    np.random.RandomState(RNG).shuffle(idx)

    return (
        x[idx].astype('float32') / 255.0,
        y[idx].astype('int32')
    )

x_train, y_train = subset(x_train_full, y_train_full, 500)
x_test, y_test = subset(x_test_full, y_test_full, 100)

print(x_train.shape, x_test.shape, 'range', x_train.min(), x_train.max())

## Step 1 — Visualize the Data

In [ ]:
fig, axes = plt.subplots(3, 6, figsize=(9, 4.7))

for ax, i in zip(axes.ravel(), range(18)):
    ax.imshow(x_train[i])
    ax.set_title(CLASSES[y_train[i]], fontsize=8)
    ax.axis('off')

plt.suptitle('CIFAR-10 subset — sample images')
plt.tight_layout()
plt.show()

counts = np.bincount(y_train, minlength=10)

for c, n in enumerate(counts):
    print(f'{CLASSES[c]:12s} {n}')

## Step 2 — Baseline CNN From Scratch

In [ ]:
def make_baseline_cnn():
    model = keras.Sequential([
        keras.Input((32, 32, 3)),

        layers.Conv2D(32, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(64, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),
        layers.MaxPooling2D(),

        layers.Conv2D(128, 3, padding='same', activation='relu'),
        layers.BatchNormalization(),

        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.3),
        layers.Dense(10, activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    return model

baseline = make_baseline_cnn()
baseline.summary()

## Train Baseline

In [ ]:
es = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=4,
    restore_best_weights=True
)

hist_base = baseline.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=25,
    batch_size=64,
    callbacks=[es],
    verbose=2
)

base_test = baseline.evaluate(x_test, y_test, verbose=0)[1]

print(f'BASELINE test accuracy = {base_test:.3f}')

## Baseline Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(hist_base.history['loss'], label='train')
axes[0].plot(hist_base.history['val_loss'], label='validation')
axes[0].set_title('Baseline Loss')
axes[0].legend()

axes[1].plot(hist_base.history['accuracy'], label='train')
axes[1].plot(hist_base.history['val_accuracy'], label='validation')
axes[1].set_title('Baseline Accuracy')
axes[1].legend()

plt.tight_layout()
plt.show()

## Step 3 — Data Augmentation

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.06),
    layers.RandomZoom(0.1)
], name='augment')

## Preview Augmentation

In [ ]:
img = x_train[1][None, ...]

fig, axes = plt.subplots(1, 5, figsize=(11, 2.6))

axes[0].imshow(img[0])
axes[0].set_title('original', fontsize=9)
axes[0].axis('off')

for ax in axes[1:]:
    augmented = data_augmentation(img, training=True)[0]
    ax.imshow(np.clip(augmented, 0, 1))
    ax.set_title('augmented', fontsize=9)
    ax.axis('off')

plt.suptitle(f'Augmentation preview — {CLASSES[y_train[1]]}')
plt.tight_layout()
plt.show()

## Step 4 — Transfer Learning: Feature Extraction

MobileNetV2 is frozen first so only the new classification head learns.

In [ ]:
IMG = 96

base = keras.applications.MobileNetV2(
    input_shape=(IMG, IMG, 3),
    include_top=False,
    weights='imagenet'
)

base.trainable = False

inputs = keras.Input((32, 32, 3))

x = data_augmentation(inputs)
x = layers.Resizing(IMG, IMG)(x)
x = layers.Rescaling(255.0)(x)
x = keras.applications.mobilenet_v2.preprocess_input(x)

x = base(x, training=False)

x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(10, activation='softmax')(x)

tl_model = keras.Model(inputs, outputs)

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print(
    'trainable params (head only):',
    sum(np.prod(w.shape) for w in tl_model.trainable_weights)
)

tl_model.summary()

## Train Feature-Extraction Model

In [ ]:
ckpt = keras.callbacks.ModelCheckpoint(
    'day20_best.keras',
    monitor='val_accuracy',
    save_best_only=True
)

es2 = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

hist_fe = tl_model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=8,
    batch_size=64,
    callbacks=[es2, ckpt],
    verbose=2
)

fe_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]

print(f'FEATURE-EXTRACTION test accuracy = {fe_test:.3f}')

## Step 5 — Fine-Tuning

Unfreeze only the upper part of MobileNetV2 and reduce the learning rate.

In [ ]:
base.trainable = True

for layer in base.layers[:-30]:
    layer.trainable = False

for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

es3 = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

hist_ft = tl_model.fit(
    x_train,
    y_train,
    validation_split=0.2,
    epochs=6,
    batch_size=64,
    callbacks=[es3, ckpt],
    verbose=2
)

ft_test = tl_model.evaluate(x_test, y_test, verbose=0)[1]

print(f'FINE-TUNED test accuracy = {ft_test:.3f}')
print(f'baseline={base_test:.3f} feat-extract={fe_test:.3f} fine-tuned={ft_test:.3f}')

## Step 6 — Confusion Matrix and Per-Class Scores

In [ ]:
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)

y_prob = tl_model.predict(x_test, verbose=0)
y_pred = y_prob.argmax(axis=1)

cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6.5, 6))

ConfusionMatrixDisplay(
    cm,
    display_labels=CLASSES
).plot(
    ax=ax,
    xticks_rotation=45,
    colorbar=False
)

plt.title('Confusion matrix — fine-tuned model')
plt.tight_layout()
plt.show()

print(
    classification_report(
        y_test,
        y_pred,
        target_names=CLASSES,
        digits=3
    )
)

## Find the Worst-Confused Pair

In [ ]:
cm_errors = cm.copy()
np.fill_diagonal(cm_errors, 0)

true_id, pred_id = np.unravel_index(
    np.argmax(cm_errors),
    cm_errors.shape
)

worst_true = CLASSES[true_id]
worst_pred = CLASSES[pred_id]
worst_count = int(cm_errors[true_id, pred_id])

print(
    f'Worst confusion: true {worst_true} -> predicted {worst_pred} '
    f'({worst_count} images)'
)

## Step 7 — Grad-CAM

Grad-CAM helps visualize which image regions influenced a prediction.

In [ ]:
grad_model = keras.Model(
    tl_model.inputs,
    [base.output, tl_model.output]
)

def gradcam(img_1, pred_index=None):

    arr = img_1[None, ...]

    with tf.GradientTape() as tape:

        conv_out, preds = grad_model(
            arr,
            training=False
        )

        if pred_index is None:
            pred_index = int(tf.argmax(preds[0]))

        channel = preds[:, pred_index]

    grads = tape.gradient(channel, conv_out)

    weights = tf.reduce_mean(
        grads,
        axis=(0, 1, 2)
    )

    cam = tf.reduce_sum(
        conv_out[0] * weights,
        axis=-1
    )

    cam = tf.maximum(cam, 0)

    cam = cam / (
        tf.reduce_max(cam) + 1e-8
    )

    return cam.numpy(), pred_index

## Display Grad-CAM for Correct Predictions

In [ ]:
correct = np.where(y_pred == y_test)[0]
show = correct[:3]

fig, axes = plt.subplots(2, 3, figsize=(9, 6))

for col, i in enumerate(show):

    cam, pidx = gradcam(x_test[i])

    cam_up = tf.image.resize(
        cam[..., None],
        (32, 32)
    ).numpy().squeeze()

    axes[0, col].imshow(x_test[i])
    axes[0, col].axis('off')
    axes[0, col].set_title(
        f'pred: {CLASSES[pidx]}',
        fontsize=9
    )

    axes[1, col].imshow(x_test[i])
    axes[1, col].imshow(
        cam_up,
        cmap='jet',
        alpha=0.45
    )
    axes[1, col].axis('off')
    axes[1, col].set_title('Grad-CAM', fontsize=9)

plt.suptitle('Where the model looks')
plt.tight_layout()
plt.show()

## Step 8 — Baseline vs Transfer Learning Results

In [ ]:
print('=== RESULTS SUMMARY ===')
print(f"{'approach':22s}{'test acc':>10s}")

for name, acc in [
    ('Baseline CNN (scratch)', base_test),
    ('Transfer: feat-extract', fe_test),
    ('Transfer: fine-tuned', ft_test)
]:
    print(f'{name:22s}{acc:10.3f}')

lift = ft_test - base_test

print(
    f'\nabsolute lift (fine-tuned - baseline) = {lift:+.3f}'
)

print(
    f'worst confusion: {worst_true} -> {worst_pred}'
)

## Step 8 — Written Reflection

**Important:** Replace the text below after running the notebook. Use your real measured numbers.

In [ ]:
reflection = f'''\nThe baseline CNN achieved {base_test:.3f} test accuracy, while the fine-tuned MobileNetV2 achieved {ft_test:.3f}, giving an absolute improvement of {lift:+.3f}. Transferrning helped because MobileNetV2 already contained useful visual features learned from ImageNet, while the baseline had to learn every feature from only 5,000 small CIFAR-10 training images. Feature extraction achieved {fe_test:.3f}, showing that the frozen pretrained features were already useful, while fine-tuning achieved {ft_test:.3f} after allowing the upper MobileNetV2 layers to adapt to CIFAR-10. The largest confusion was true {worst_true} being predicted as {worst_pred}, which suggests that these classes share difficult low-resolution visual features. After inspecting Grad-CAM, replace this sentence with whether the model focused mainly on the object or relied on background regions. As a next experiment, I would test stronger augmentation or a different number of unfrozen MobileNetV2 layers while keeping the test set unchanged.\n'''

print(reflection)

## Save Final Model

In [ ]:
tl_model.save('day20_final_finetuned.keras')

print('Saved final model -> day20_final_finetuned.keras')

## Final Validation

In [ ]:
assert x_train.shape == (5000, 32, 32, 3)
assert x_test.shape == (1000, 32, 32, 3)
assert np.all(np.bincount(y_train, minlength=10) == 500)
assert y_prob.shape == (1000, 10)
assert cm.shape == (10, 10)

print('All Day 20 capstone validation checks passed.')

# Submission Checklist

- Run every cell from top to bottom.
- Confirm the sample-image plot appears.
- Confirm augmentation examples appear.
- Confirm baseline CNN training completes.
- Confirm MobileNetV2 feature extraction completes.
- Confirm fine-tuning completes.
- Confirm confusion matrix appears.
- Confirm per-class report prints.
- Confirm Grad-CAM images appear.
- Edit the Step 8 reflection using your actual results.
- Save this notebook as `day20_capstone.ipynb` with outputs visible.